# Track Deleted Works (oxjob #784)

Maintains the durable ledger of works that have disappeared from
`openalex.works.openalex_works`. Runs right after Guardrails, so a run that
loses works pathologically is stopped before it can ledger the loss.

Two tables:

- **`openalex.works.deleted_works`** — the ledger. One row per (work, deletion):
  `work_id` (BIGINT), `deleted_date` (first date the work was detected gone),
  `deleted_at` (timestamp), `es_deleted_at` (stamped by
  `notebooks/elastic/delete_works` once the ES doc is removed; NULL = ES delete
  still pending). Consumed by the ES delete task and by the daily snapshot's
  `deleted.csv` export.
- **`openalex.works.openalex_works_id_snapshot`** — the id set of
  `openalex_works` as of the last successful run of this notebook. Tonight's
  deletions = snapshot ∖ current. Rewritten at the end of every run, only after
  the ledger insert succeeded. First run seeds it and ledgers nothing.

**Resurrections**: a work that reappears in `openalex_works` is dropped from the
ledger (it is alive; the CSV must not list it). Its ES doc restores itself
without help: the vanish deleted its `openalex_works_hash` row (NOT MATCHED BY
SOURCE), so reappearing re-inserts the hash row with a fresh `updated_date`,
which the nightly 2-day incremental ES sync picks up.

**Guard**: aborts (no ledger insert, snapshot untouched) if `openalex_works` is
empty or tonight's deletions exceed `guard_fraction` (default 0.5%) of the live
count. Guardrails already fails the run upstream at >2M works lost, so tripping
this guard means something slipped past it — investigate before overriding.
Sanctioned mass deletions (#765 drain waves) run with the
`deleted_works_guard_override` job parameter set to `true`.


In [0]:
dbutils.widgets.text("guard_override", "false")
dbutils.widgets.text("guard_fraction", "0.005")
dbutils.widgets.text("env_suffix", "")

GUARD_OVERRIDE = dbutils.widgets.get("guard_override").lower() == "true"
GUARD_FRACTION = float(dbutils.widgets.get("guard_fraction"))
ENV_SUFFIX = dbutils.widgets.get("env_suffix")

CATALOG = f"openalex{ENV_SUFFIX}"
WORKS = f"{CATALOG}.works.openalex_works"
LEDGER = f"{CATALOG}.works.deleted_works"
ID_SNAPSHOT = f"{CATALOG}.works.openalex_works_id_snapshot"

print(f"guard_override: {GUARD_OVERRIDE}")
print(f"guard_fraction: {GUARD_FRACTION}")
print(f"works: {WORKS}")


In [0]:
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {LEDGER} (
    work_id BIGINT NOT NULL,
    deleted_date DATE NOT NULL,
    deleted_at TIMESTAMP NOT NULL,
    es_deleted_at TIMESTAMP
)
""")


In [0]:
resurrected = spark.sql(
    f"DELETE FROM {LEDGER} WHERE work_id IN (SELECT id FROM {WORKS})"
).collect()[0].num_affected_rows
print(f"Resurrected works removed from ledger: {resurrected:,}")


In [0]:
current_count = spark.sql(f"SELECT COUNT(*) AS cnt FROM {WORKS}").collect()[0].cnt
print(f"Current {WORKS} count: {current_count:,}")

if not spark.catalog.tableExists(ID_SNAPSHOT):
    if current_count == 0:
        raise Exception(f"ABORT: {WORKS} is empty; refusing to seed {ID_SNAPSHOT} from it.")
    spark.sql(f"CREATE TABLE {ID_SNAPSHOT} AS SELECT id FROM {WORKS}")
    print(f"First run: seeded {ID_SNAPSHOT} with {current_count:,} ids; no deletions ledgered.")
else:
    gone_df = spark.sql(f"""
        SELECT s.id AS work_id
        FROM {ID_SNAPSHOT} s
        LEFT ANTI JOIN {WORKS} w ON s.id = w.id
    """)
    gone_count = gone_df.count()
    print(f"Works gone since last run: {gone_count:,}")

    if current_count == 0:
        raise Exception(
            f"ABORT: {WORKS} has 0 rows; every id would ledger as deleted. "
            "Upstream build failed in a way Guardrails missed. Ledger and snapshot unchanged."
        )
    if gone_count > GUARD_FRACTION * current_count and not GUARD_OVERRIDE:
        raise Exception(
            f"ABORT: {gone_count:,} works gone (> {GUARD_FRACTION:.2%} of {current_count:,} live). "
            "Ledger and snapshot unchanged, so tonight's diff is preserved for inspection. "
            "If this is a sanctioned mass deletion, re-run with deleted_works_guard_override=true."
        )

    gone_df.createOrReplaceTempView("gone_work_ids")
    inserted = spark.sql(f"""
        INSERT INTO {LEDGER}
        SELECT g.work_id, current_date(), current_timestamp(), NULL
        FROM gone_work_ids g
        LEFT ANTI JOIN {LEDGER} l ON g.work_id = l.work_id
    """).collect()[0].num_inserted_rows
    print(f"Ledgered {inserted:,} deletions (dated {spark.sql('SELECT current_date() AS d').collect()[0].d}).")

    spark.sql(f"CREATE OR REPLACE TABLE {ID_SNAPSHOT} AS SELECT id FROM {WORKS}")
    print(f"Snapshot rewritten with {current_count:,} ids.")

pending = spark.sql(
    f"SELECT COUNT(*) AS cnt FROM {LEDGER} WHERE es_deleted_at IS NULL"
).collect()[0].cnt
total = spark.sql(f"SELECT COUNT(*) AS cnt FROM {LEDGER}").collect()[0].cnt
print(f"Ledger: {total:,} total deletions, {pending:,} pending ES delete.")
